In [22]:
import os
from pathlib import Path
import sys

import pandas as pd

_root_override = os.environ.get("THESIS_REPO_ROOT")
if _root_override:
    _bootstrap_root = Path(_root_override).expanduser().resolve()
else:
    _bootstrap_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / ".git").exists()), None)
if _bootstrap_root is None:
    raise RuntimeError("Repository root not found; set THESIS_REPO_ROOT or start Jupyter inside the repository.")
if str(_bootstrap_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_root))

from src.thesis_pipeline.paths import DATA_PROCESSED_DIR, REPO_ROOT

PROJECT_DIR = REPO_ROOT
DATA_PATH = DATA_PROCESSED_DIR

df02funnel = pd.read_csv(DATA_PATH / "02_mnl_estimation_funnel.csv", index_col=0)
df02wide = pd.read_csv(DATA_PATH / "02_trip_vehicle_wide.csv", index_col=0)

print("Dataframe shape:", df02funnel.shape)
df02funnel.head(8)

Dataframe shape: (8, 3)


,n_households,n_persons,n_trip_rows
step,,,
1. all trip rows,172347,305206,1087393
"2. + W_WAUTO in {1,2,3}",67776,109534,291426
3. + Pkw driver trip,65497,81061,219761
4. + chosen car matched in Autos,65497,81061,219761
5. + household has at least 2 model-usable alternatives in Autos,35908,49456,136687
6. + merged HH/person variables available,35908,49456,136687
7. + all selected trial-model variables valid,31124,41616,114058
8. final wide MNL sample,31124,41616,114058


Final sample:
114,058 driver-trip observations from 41,616 persons in 31,124 households.
89,564 observations have two model-usable vehicle alternatives, and 24,494 observations have three.

In [23]:
print("Dataframe shape:", df02wide.shape)
print(df02wide.columns.tolist())


Dataframe shape: (114058, 33)
['P_ID', 'W_ID', 'choice', 'n_valid_alternatives', 'av_1', 'av_2', 'av_3', 'W_WAUTO', 'pkw_fmf', 'zweck', 'wegkm_imp', 'wegmin_imp2', 'ST_WOTAG', 'ST_MONAT', 'H_ANZAUTO', 'hhgr_gr', 'oek_status', 'RegioStaR7', 'HP_SEX', 'alter_gr5', 'Taet', 'antrieb_1', 'A_BAUJ_1', 'seg_kba_gr_1', 'status_1', 'antrieb_2', 'A_BAUJ_2', 'seg_kba_gr_2', 'status_2', 'antrieb_3', 'A_BAUJ_3', 'seg_kba_gr_3', 'status_3']


In [24]:
hh_day_use = (
    df02wide.groupby("H_ID")
    .agg(
        n_driver_car_trips=("W_ID", "count"),
        n_drivers=("P_ID", "nunique"),
        n_used_cars=("choice", "nunique"),
        n_valid_alternatives=("n_valid_alternatives", "max"),
        total_km=("wegkm_imp", "sum"),
        mean_trip_km=("wegkm_imp", "mean"),
    )
    .reset_index()
)

hh_day_use["trips_per_used_car"] = (
    hh_day_use["n_driver_car_trips"] / hh_day_use["n_used_cars"]
)

hh_day_use["n_driver_car_trips"].describe()

count    31124.000000
mean         3.664632
std          2.367712
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         24.000000
Name: n_driver_car_trips, dtype: float64

In [25]:
hh_day_use["n_drivers"].value_counts()

n_drivers
1    21384
2     9026
3      676
4       38
Name: count, dtype: int64

In [26]:
hh_day_use["n_driver_car_trips"].value_counts().sort_index()

n_driver_car_trips
1      3481
2     10349
3      3201
4      5793
5      2171
6      2521
7      1162
8      1043
9       532
10      384
11      195
12      136
13       67
14       33
15       29
16       13
17        7
18        1
20        3
21        1
22        1
24        1
Name: count, dtype: int64

In [27]:
pd.crosstab(
    hh_day_use["n_valid_alternatives"],
    hh_day_use["n_used_cars"],
    margins=True
)

n_used_cars,1,2,3,All
n_valid_alternatives,,,,
2,18132,6888,0,25020
3,3790,1780,534,6104
All,21922,8668,534,31124


In [ ]:
car_day_use = (
    df02wide.groupby(["H_ID", "choice"])
    .agg(
        n_trips_by_car=("W_ID", "count"),
        n_drivers_by_car=("P_ID", "nunique"),
        km_by_car=("wegkm_imp", "sum"),
        mean_trip_km_by_car=("wegkm_imp", "mean"),
    )
    .reset_index()
    .rename(columns={"choice": "A_ID"})
)


car_day_use["n_trips_by_car"].value_counts().sort_index()

n_trips_by_car
1      5275
2     19316
3      4937
4      6516
5      1992
6      1588
7       603
8       346
9       159
10       73
11       35
12       13
13        4
14        1
15        1
16        1
Name: count, dtype: int64

In [31]:
hh_day_use["n_driver_car_trips"].value_counts().sum()

np.int64(31124)

In [30]:
car_day_use["n_trips_by_car"].describe()

count    40860.000000
mean         2.791434
std          1.557693
min          1.000000
25%          2.000000
50%          2.000000
75%          4.000000
max         16.000000
Name: n_trips_by_car, dtype: float64

In [19]:
car_day_use.groupby("A_ID")["n_trips_by_car"].describe()

,count,mean,std,min,25%,50%,75%,max
A_ID,,,,,,,,
1,22288.0,2.831210,1.604652,1.0,2.0,2.0,4.0,15.0
2,16448.0,2.763801,1.507351,1.0,2.0,2.0,4.0,16.0
3,2124.0,2.588041,1.414636,1.0,2.0,2.0,3.0,12.0
